# Setup

In [ ]:
%%capture
# 1. Install dependencies
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

# Extracting information without structured outputs


In [12]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae.
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 on the independent test set.
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
"""

resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Extract the title, author, organism, method, and metric from the provided abstract.",
        },
        {"role": "user", "content": abstract},
    ]
)

raw_response = resp.choices[0].message.content

print(raw_response) # Hard to parse and to work with!


Title: A novel deep learning approach for predicting protein-protein interactions in Saccharomyces cerevisiae  
Author: Not provided in the abstract  
Organism: Saccharomyces cerevisiae  
Method: Graph neural networks  
Metric: AUC-ROC score of 0.92  


In [9]:
lines = raw_response.split('\n')
metric_line = next((line for line in lines if line.startswith('Metric:')), None)

if metric_line:
    parsed_metric = metric_line.replace('Metric:', '').strip()
    print(f"Parsed Metric from raw response: {parsed_metric}")
else:
    print("Metric not found in raw response.")

Parsed Metric from raw response: AUC-ROC score of 0.92


# Structured output with the raw completions API

In [7]:
from openai import OpenAI
import os
import json

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae. 
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 on the independent test set. 
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
"""

resp = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Return JSON with keys: title, author, organism, method, metric.", #We have to specify the keys we want to extract from the abstract
        },
        {"role": "user", "content": abstract},
    ],
    response_format={"type": "json_object"},
)

raw = resp.choices[0].message.content
data = json.loads(raw)

print(f"Title: {data.get("title")}")
print(f"Author: {data.get("author")}")
print(f"Organism: {data.get("organism")}")
print(f"Method: {data.get("method")}")
print(f"Metric: {data.get("metric")}")
print(data)


Title: A Deep Learning Approach for Predicting Protein-Protein Interactions in Saccharomyces cerevisiae
Author: Not specified
Organism: Saccharomyces cerevisiae
Method: Graph Neural Networks
Metric: AUC-ROC score of 0.92
{'title': 'A Deep Learning Approach for Predicting Protein-Protein Interactions in Saccharomyces cerevisiae', 'author': 'Not specified', 'organism': 'Saccharomyces cerevisiae', 'method': 'Graph Neural Networks', 'metric': 'AUC-ROC score of 0.92'}


In [8]:
# We may have intended a numeric value
baseline = 0.5
data.get("metric") - baseline 

TypeError: unsupported operand type(s) for -: 'str' and 'float'

In [9]:
# We might get different data types than we expect
author = data.get("author")
if author:
    print(f'This paper was authored by {author}')
else:
    # Some handling for missing data
    print("No author information available.")
    pass


This paper was authored by Not specified


# Structured output with PydanticAI

In [10]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

In [11]:
from pydantic_ai import Agent
from pydantic import BaseModel

class PaperSummary(BaseModel):
    title: str | None
    author: str | None
    organism: str | None
    method: str | None
    metric: float | None

agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=PaperSummary)
result = agent.run_sync(f"Abstract to parse: {abstract}")
structured_output = result.output
print(type(structured_output))
print(structured_output.metric) #Actually returns a float number

<class '__main__.PaperSummary'>
0.92


In [12]:
print(structured_output.model_dump_json(indent=2))
# Output can still be hallucinated or output UNKNOWN instead of None

{
  "title": "Deep Learning Approach for Predicting Protein-Protein Interactions in Saccharomyces cerevisiae",
  "author": "<UNKNOWN>",
  "organism": "Saccharomyces cerevisiae",
  "method": "Graph neural networks",
  "metric": 0.92
}


# Exercise

In [13]:
#TODO create an LLM-call that will extract the protocol name, published year, and expressed genes
text = """
In this paper, we present a novel protocol for isolating and culturing human embryonic stem cells (hESCs).
The protocol was first published in 2010 and has since been widely adopted in the field of stem cell research.
The protocol involves the use of feeder-free conditions and the expression of key pluripotency markers such as OCT4, SOX2, and NANOG.
"""
